## Part 3: Using Vision LLMs for Structured Data Scraping

We will work on the scanned forms found in [this California campaign finance PDF](./samples/ca-campfin-contributions.pdf)!

Take a look at that PDF: Can you select the text? 

No! Ok, what if we OCR'ed it?

Well it would include ALL the text in that page, including the title, form fields, descriptions... we don't care about that. We want the values in the boxes only! And some of these are checkmarks... How do you OCR checkmarks?

This historically has been a hard, expensive problem to solve. But the advancedment of LLMs in the past few months means that this is now MUCH easier and cheaper to do, with pretty great results!!!

Let's give it a go!

In [ ]:
from part3_utils import parse_page

import json
from pydantic import BaseModel
from typing import List, Optional
import os
from pathlib import Path

This will offer a popup to include an OpenRouter API key - take it from the slides!

In [ ]:
# Ask you for the API key on first run, and save it to a file for subsequent runs
if Path(".api_key").exists():
    os.environ["OPENROUTER_API_KEY"] = Path(".api_key").read_text().strip()
else:
  api_key = input()
  Path(".api_key").write_text(api_key.strip())
  os.environ["OPENROUTER_API_KEY"] = api_key.strip()

## Using OpenRouter Structured Ouputs with Gemini 

LLMs typically responding in English, in a chat-bot sort of way. 

We don't want that here, we want then to respond in a machine-readable format that we can process with code. Ideally, we want them to respond with JSON, in a schema that we define.

This is exactly what [structured outputs with LLMs](https://openrouter.ai/docs/guides/features/structured-outputs) does! We define a schema for how the LLM should respond, and it'll resond in that format!

See the [`part3_utils.py`](./part3_utils.py) file for doing this in Python with OpenRouter. It requires a LLM capability of reading images (a "vision LLM"). Here we use Google Gemini 3 Flash, but OpenRouter allows us to test with other models.

In [ ]:

# Instructing the model on how to parse the data into structured JSON.
PROMPT = """
  Parse the given CA Form 460 Schedule into JSON.

  For any dates, provide the date in the format YYYY-MM-DD.
  If a field is not applicable, use `null` for the value.
"""

# A Pydantic model representing the structure of the data we want to extract from the form.
# Here is the schema we want from a single row from CA's Form 460 Schedule A
class Form460ScheduleALineItem(BaseModel):
    date_received: str
    full_name: str
    city: str
    state: str
    zipcode: str
    contributor_code: str
    occupation: str
    employer: str
    amount_this_period: float
    amount_cumulative_calendar_year: float
    amount_per_election_code: Optional[str]
    amount_per_election: Optional[float]


# A Pydantic model representing the entire Schedule A, which consists of multiple line items.
class Form460ScheduleA(BaseModel):
    line_items: List[Form460ScheduleALineItem]


In [ ]:
! rm -rf campfin_pages || true
! mkdir campfin_pages
! uv run pypdfium2 extract-images samples/ca-campfin-contributions.pdf -o ./campfin_pages

In [ ]:
image = Path("campfin_pages/ca-campfin-contributions_02_1.png").read_bytes()
parsed = parse_page(
    prompt=PROMPT,
    schema=Form460ScheduleA,
    image=image,
)
print(json.dumps(parsed, indent=2))

## OK that's cool and all but what about handwriting??

Let's try it with [this handwritten schedule A](./samples/ca-campfin-contributions-handwritten.pdf)!

In [ ]:
! rm -rf handwritten_pages || true
! mkdir handwritten_pages
! uv run pypdfium2 extract-images samples/ca-campfin-contributions-handwritten.pdf -o ./handwritten_pages

In [ ]:
image = Path("handwritten_pages/ca-campfin-contributions-handwritten_1_1.png").read_bytes()
parsed = parse_page(
    prompt=PROMPT,
    schema=Form460ScheduleA,
    image=image,
)
print(json.dumps(parsed, indent=2))